# Module 2.3 — A2A: protocol-enforced message isolation

Module 2.2 §6 ended on a cliff:

> *The architectural fix is not "write a better prompt". It is **message isolation**: give each agent only the inputs its role actually needs, via typed messages with explicit sender/receiver fields. That is what the A2A protocol (Module 2.3) provides.*

That claim was written on the strength of a working prompt-patched Skeptic. It is time to honour it.

This module keeps **every** ingredient of 2.2 — same model (`qwen2.5:7b`), same `calculator_tool`, same three Onsager temperatures, same `Theorist` / `Skeptic` / `Judge` agents with identical `role` / `goal` / `backstory`. One thing changes: the Crew's `Process.sequential` is replaced by a minimal **in-process message bus**. Each agent runs inside its own single-agent Crew whose task description is assembled *only* from the agent's typed inbox. Isolation stops being a prompt discipline you hope the model follows; it becomes a line of Python you can read and a `assert` you can run.

## 1. What we're proving

Three linked claims, each testable:

1. **Isolation becomes inspectable.** In 2.2, "the Judge sees only the Skeptic's table" was a sentence we wrote in a prompt and hoped the runtime honoured. In 2.3 we print the Judge's inbox and `assert` its contents. If the contract is violated, the notebook raises.
2. **The Skeptic's discipline no longer depends on prompt gymnastics.** 2.2 needed an aggressive "pretend the Theorist's table is not visible" instruction + SCRATCH-lines-before-table to beat 7B sycophancy. Under A2A, the Skeptic's input is *literally* just the claim table (not "and here's what a reasonable answer looks like" leaking through shared context). The prompt can be shorter.
3. **Same task, same agents, same model — different guarantee.** If the end-to-end verdict matches 2.2's (the Skeptic catches the Theorist's hallucination, the Judge rules `RECONCILE`), we have shown the protocol rearrangement is a drop-in replacement — you do not have to redesign your agents to adopt A2A.

What we are *not* doing:

- We are not using `python-a2a`, Google's reference SDK. That is a real, HTTP-based library; it shines when agents run in separate processes or on separate machines. This course runs a 7B on a laptop. ~20 lines of in-process Python make the *ideas* visible without the networking.
- We are not introducing a new physics scenario. Same task as 2.2 — the point is the A/B.

The §7 exercises include a pointer to `python-a2a` so you can take the pattern off the laptop when you need to.

## 2. Setup

Boilerplate-identical to 2.2. If you just ran 2.2, the kernel already has everything — skip to §3.

In [1]:
# Same OTEL-disable + warnings-filter boilerplate as 2.0 / 2.1 / 2.2.
import os
os.environ["OTEL_SDK_DISABLED"]        = "true"
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"

import warnings
warnings.filterwarnings("ignore")

from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(
    model="ollama/qwen2.5:7b",
    base_url="http://localhost:11434",
    temperature=0.0,
)
print(f"LLM: {llm.model}")

LLM: ollama/qwen2.5:7b


In [2]:
# calculator_tool, re-decorated for CrewAI. Verbatim from Module 2.2.
import ast, math, operator as op
from crewai.tools import tool as ct_tool

_ALLOWED_BINOPS   = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
                     ast.Div: op.truediv, ast.Pow: op.pow,
                     ast.Mod: op.mod, ast.FloorDiv: op.floordiv}
_ALLOWED_UNARYOPS = {ast.UAdd: op.pos, ast.USub: op.neg}
_ALLOWED_NAMES    = {"pi": math.pi, "e": math.e, "sqrt": math.sqrt,
                     "log": math.log, "log10": math.log10, "exp": math.exp,
                     "sin": math.sin, "cos": math.cos, "tan": math.tan,
                     "sinh": math.sinh, "cosh": math.cosh, "tanh": math.tanh,
                     "abs": abs}

def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
        v = _ALLOWED_NAMES[node.id]
        return v if callable(v) else float(v)
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        return _ALLOWED_BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        fn = _ALLOWED_NAMES.get(node.func.id)
        if callable(fn) and not node.keywords:
            return fn(*[_safe_eval(a) for a in node.args])
    raise ValueError(f"disallowed expression: {ast.dump(node)}")

@ct_tool("calculator_tool")
def calculator_tool(expression: str) -> float:
    """Evaluate a math expression. Supports +,-,*,/,**, sqrt, log, exp, sin/cos/tan, sinh/cosh/tanh, pi, e.

    Args:
        expression: a math expression, e.g. '2 / log(1 + sqrt(2))' for Onsager T_c.
    """
    return float(_safe_eval(ast.parse(expression, mode="eval")))

print("calculator_tool check: T_c =", calculator_tool.run(expression="2 / log(1 + sqrt(2))"))

Using Tool: calculator_tool
calculator_tool check: T_c = 2.269185314213022


## 3. The message bus

Two data structures and two methods. Read them carefully — this is the entire A2A surface we need.

```
Message          typed envelope:  sender, receiver, kind, payload, id
MessageBus       in-process router with one inbox per agent + a global log
  .send(msg)     deliver to receiver's inbox, append to log
  .inbox(name)   filterable view onto an agent's queue
```

Why these specific fields:

- **`kind`** is how the Judge knows it is looking at a `ComparisonTable` and not a `ClaimToReview`. It is the machine-readable contract. Free-form prose has no such handle — which is why 2.2's Judge had to *pattern-match* the `agree?` column from an untyped string.
- **`sender` / `receiver`** are what turn "context" into "routed mail". In 2.2, context was whatever the framework passed; in 2.3, it is whatever `bus.inbox('judge')` returns, by definition.
- **`id`** lets downstream messages reference upstream ones via `parent_id` (we don't exercise it here, but the exercises do).

No HTTP. No ports. No serialisation. This runs in one Python process. The point of the module is the *shape* — `python-a2a`'s `A2AClient` / `A2AServer` carry the same shape over real networking when you need to cross process boundaries (see §8).

In [3]:
from dataclasses import dataclass, field
from collections import defaultdict
from typing import Optional
import uuid

@dataclass
class Message:
    sender: str
    receiver: str
    kind: str          # "ClaimToReview" | "ComparisonTable" | "Verdict"
    payload: str       # content of the message (a markdown table, a verdict, ...)
    parent_id: Optional[str] = None
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])

    def short(self) -> str:
        return f"[{self.id}] {self.sender:>10s} -> {self.receiver:<10s}  kind={self.kind:<16s} ({len(self.payload)}c)"


class MessageBus:
    """In-process router. One inbox per agent name + a global ordered log."""

    def __init__(self):
        self._inboxes: dict[str, list[Message]] = defaultdict(list)
        self.log: list[Message] = []

    def send(self, msg: Message) -> None:
        self._inboxes[msg.receiver].append(msg)
        self.log.append(msg)

    def inbox(self, name: str, kind: Optional[str] = None) -> list[Message]:
        msgs = self._inboxes[name]
        return [m for m in msgs if kind is None or m.kind == kind]


# Smoke-test the bus before we plug agents into it.
_bus = MessageBus()
_bus.send(Message(sender="alice", receiver="bob", kind="Ping", payload="hello"))
assert len(_bus.inbox("bob")) == 1
assert _bus.inbox("bob", kind="Pong") == []
print("bus smoke test OK:", _bus.log[0].short())

bus smoke test OK: [ce7f84da]      alice -> bob         kind=Ping             (5c)


## 4. Same three agents as Module 2.2

Identical `role`, `goal`, `backstory`, `tools`, `max_iter`. This is the "only the protocol changes" part of the experiment. Copy-paste from 2.2 on purpose — if you open both notebooks side-by-side, the block below should match Module 2.2 §3 character-for-character except for one line we'll call out in the next cell.

In [4]:
theorist = Agent(
    role="Theoretical Physicist",
    goal=(
        "Compute Onsager's |m|(T) at three temperatures using calculator_tool and return a "
        "Markdown table. Report numbers, not methods."
    ),
    backstory=(
        "You know Onsager 1944. T_c = 2 / ln(1 + sqrt(2)). For T < T_c, "
        "|m|(T) = (1 - sinh(2/T)**(-4))**(1/8). For T >= T_c it is 0. "
        "You always compute with calculator_tool -- never by inspection."
    ),
    tools=[calculator_tool],
    llm=llm,
    allow_delegation=False,
    max_iter=3,
    verbose=True,
)

skeptic = Agent(
    role="Peer Reviewer",
    goal=(
        "Independently re-derive |m|(T) with calculator_tool at the same temperatures the "
        "Theorist was asked about, then compare row-by-row with the Theorist's claimed values "
        "and report disagreements."
    ),
    backstory=(
        "You do not trust numbers you did not compute yourself. For every T you verify, you "
        "call calculator_tool -- you never copy the Theorist's values into your own column. "
        "You are not writing prose; you produce a comparison table."
    ),
    tools=[calculator_tool],
    llm=llm,
    allow_delegation=False,
    max_iter=5,
    verbose=True,
)

judge = Agent(
    role="Journal Editor",
    goal=(
        "Read the Peer Reviewer's comparison table and issue ONE verdict line from a fixed "
        "vocabulary. Do not re-derive. Do not restate."
    ),
    backstory=(
        "You do not compute. You read one document -- the Peer Reviewer's comparison table -- "
        "and decide: ACCEPT_THEORIST (all rows agree), ACCEPT_SKEPTIC (no row agrees), or "
        "RECONCILE (mixed). Your output is a single line in a fixed format."
    ),
    tools=[],
    llm=llm,
    allow_delegation=False,
    max_iter=2,
    verbose=True,
)

print("Agents built (identical to Module 2.2):")
for a in (theorist, skeptic, judge):
    print(f"  - {a.role:25s}  tools={[t.name for t in a.tools] if a.tools else '[]'}")

Agents built (identical to Module 2.2):
  - Theoretical Physicist      tools=['calculator_tool']
  - Peer Reviewer              tools=['calculator_tool']
  - Journal Editor             tools=[]


## 5. Three turn functions, one per role

Each turn function has the same shape:

1. Read the role's inbox (of an expected `kind`).
2. Build a task description from **only the message payload** — no shared context, no upstream table hiding in a system prompt.
3. Run a single-agent Crew to execute that one task. The agent sees exactly the description we pass.
4. Package the output as a typed `Message` and put it on the bus.

The key surgical move is `a2a_run`: it invokes an agent via a fresh `Crew(agents=[agent], tasks=[task])`, so there is no `Process.sequential` chain carrying hidden state between steps. Each call is hermetic.

Note the Skeptic's task description below: it is *shorter* than 2.2's. We no longer need the "pretend the Theorist's table is not visible" clause, because under A2A the Theorist's full table is not there to pretend about — only its rendered payload, delivered explicitly as the thing to compare against. We **keep** the SCRATCH-lines discipline: that guards against the 7B tool-call-skipping failure, which is a model behaviour independent of the isolation story.

In [5]:
from crewai import Task as _Task, Crew as _Crew, Process as _Process
# (aliased imports only for clarity -- these are the same Task/Crew already imported)

TEMPS = [2.00, 2.20, 2.50]

def a2a_run(agent: Agent, description: str, expected_output: str) -> str:
    """Run a single agent in its own one-task Crew. No shared context, no chain.

    This is the entire protocol-level isolation mechanism: each agent's invocation
    sees only `description` and `expected_output`. Nothing else leaks in.
    """
    task = _Task(description=description, expected_output=expected_output, agent=agent)
    crew = _Crew(agents=[agent], tasks=[task], process=_Process.sequential, verbose=False)
    return str(crew.kickoff())


# -- Theorist --
THEORIST_DESC = (
    f"Compute the Onsager spontaneous magnetisation |m|(T) at T = {TEMPS}. "
    "T_c = 2/ln(1+sqrt(2)) ~= 2.2692. For T < T_c, use calculator_tool to evaluate "
    "(1 - sinh(2/T)**(-4))**(1/8). For T >= T_c, the value is 0 (no tool call needed). "
    "Do NOT compute any value from memory. Procedure (follow exactly, in this order):\n"
    "  step 1: calculator_tool(expression='(1 - sinh(2/2.00)**(-4))**(1/8)')  -> record as m1\n"
    "  step 2: calculator_tool(expression='(1 - sinh(2/2.20)**(-4))**(1/8)')  -> record as m2\n"
    "  step 3: T=2.50 is above T_c, so m3 = 0\n"
    "Return a Markdown table with exactly three data rows, columns 'T' and '|m|_theorist', "
    "four decimals using m1, m2, m3."
)
THEORIST_EXP = (
    "A Markdown table with header `| T | |m|_theorist |` and exactly three data rows "
    "for T = 2.00, 2.20, 2.50. No prose. No code. Just the table."
)


# -- Skeptic template (note: {claim_payload} is the ONLY external input) --
SKEPTIC_TEMPLATE = (
    "You have received a `ClaimToReview` A2A message. Its payload is the Theorist's "
    "claimed table, reproduced below between the markers:\n"
    "--- BEGIN CLAIM PAYLOAD ---\n"
    "{claim_payload}\n"
    "--- END CLAIM PAYLOAD ---\n"
    "\n"
    f"Step A (MUST use calculator_tool): compute |m|(T) at T in {TEMPS}. Procedure:\n"
    "  A.1: calculator_tool(expression='(1 - sinh(2/2.00)**(-4))**(1/8)')  -> mA\n"
    "  A.2: calculator_tool(expression='(1 - sinh(2/2.20)**(-4))**(1/8)')  -> mB\n"
    "  A.3: T=2.50 is above T_c=2.2692, so mC = 0 (no tool call needed)\n"
    "After each tool call in A.1 and A.2, write ONE scratch line in your output showing "
    "the EXACT tool result (before the comparison table), in this format:\n"
    "   SCRATCH: mA = <exact tool output at T=2.00>\n"
    "   SCRATCH: mB = <exact tool output at T=2.20>\n"
    "   SCRATCH: mC = 0 (no call; T=2.50 > T_c)\n"
    "Step B: build ONE comparison Markdown table with columns:\n"
    "  'T'             -- the temperature,\n"
    "  '|m|_theorist'  -- the value from the claim payload above,\n"
    "  '|m|_skeptic'   -- your mA / mB / mC, rounded to four decimals,\n"
    "  'agree?'        -- YES if |theorist - skeptic| < 0.001, otherwise NO.\n"
    "Your final output is the three SCRATCH lines followed by the comparison table, "
    "and nothing else."
)
SKEPTIC_EXP = (
    "Three SCRATCH lines (SCRATCH: mA = ..., SCRATCH: mB = ..., SCRATCH: mC = 0 ...) "
    "followed by a single Markdown comparison table with header "
    "`| T | |m|_theorist | |m|_skeptic | agree? |` and exactly three data rows. "
    "No prose, no code, no other tables."
)


# -- Judge template --
JUDGE_TEMPLATE = (
    "You have received a `ComparisonTable` A2A message. Its payload is reproduced below "
    "between the markers. It is the ONLY content you have access to; do NOT imagine any "
    "other inputs.\n"
    "--- BEGIN COMPARISON PAYLOAD ---\n"
    "{comparison_payload}\n"
    "--- END COMPARISON PAYLOAD ---\n"
    "\n"
    "Do NOT re-derive any number. Do NOT call any tool. Inspect the 'agree?' column "
    "only and return exactly ONE line in this fixed format:\n"
    "  VERDICT: <LABEL>. <one sentence rationale>\n"
    "Decision rule for <LABEL>:\n"
    "  - ACCEPT_THEORIST  if every row's agree? is YES.\n"
    "  - ACCEPT_SKEPTIC   if every row's agree? is NO.\n"
    "  - RECONCILE        if the column is mixed (some YES and some NO).\n"
    "Rationale: how many rows agreed, how many disagreed."
)
JUDGE_EXP = (
    "A single line: `VERDICT: <ACCEPT_THEORIST|ACCEPT_SKEPTIC|RECONCILE>. <rationale>.` "
    "No tables, no preamble, no code."
)


def theorist_turn(bus: MessageBus) -> None:
    claim = a2a_run(theorist, THEORIST_DESC, THEORIST_EXP)
    bus.send(Message(
        sender="theorist", receiver="skeptic",
        kind="ClaimToReview", payload=claim,
    ))


def skeptic_turn(bus: MessageBus) -> None:
    msgs = bus.inbox("skeptic", kind="ClaimToReview")
    assert len(msgs) == 1, f"skeptic expected exactly 1 ClaimToReview, got {len(msgs)}"
    claim_msg = msgs[0]
    desc = SKEPTIC_TEMPLATE.format(claim_payload=claim_msg.payload)
    comparison = a2a_run(skeptic, desc, SKEPTIC_EXP)
    bus.send(Message(
        sender="skeptic", receiver="judge",
        kind="ComparisonTable", payload=comparison,
        parent_id=claim_msg.id,
    ))


def judge_turn(bus: MessageBus) -> None:
    msgs = bus.inbox("judge", kind="ComparisonTable")
    assert len(msgs) == 1, f"judge expected exactly 1 ComparisonTable, got {len(msgs)}"
    comp_msg = msgs[0]
    desc = JUDGE_TEMPLATE.format(comparison_payload=comp_msg.payload)
    verdict = a2a_run(judge, desc, JUDGE_EXP)
    bus.send(Message(
        sender="judge", receiver="user",
        kind="Verdict", payload=verdict,
        parent_id=comp_msg.id,
    ))


print("Three turn functions ready. Temperatures under test:", TEMPS)

Three turn functions ready. Temperatures under test: [2.0, 2.2, 2.5]


## 6. Run the scenario, then verify the isolation contract

Three sequential turns against the bus, then the key pedagogical moment: we print the full message log, print every agent's inbox, and `assert` three contract invariants:

1. The **Skeptic** received exactly one `ClaimToReview` from the Theorist. Nothing else.
2. The **Judge** received exactly one `ComparisonTable` from the Skeptic. No `ClaimToReview` leaked through.
3. The Judge's inbox payload *does* mention the Theorist's numbers — but **only as a column of the comparison table**, not as a separate claim document. (We don't assert this in code because it's not a clean textual invariant, but the printed inbox will show it.)

If any assertion fails, the notebook raises. That failure mode is the point: the contract is enforceable, not aspirational.

In [6]:
import time

bus = MessageBus()

t0 = time.time()
theorist_turn(bus)
skeptic_turn(bus)
judge_turn(bus)
dt = time.time() - t0

print(f"\n=== A2A run finished in {dt:.1f}s ===\n")
print("--- Bus log (all messages, in order) ---")
for m in bus.log:
    print("  ", m.short())

print("\n--- Inboxes (per receiver) ---")
for name in ["skeptic", "judge", "user"]:
    print(f"\n  inbox('{name}'):")
    for m in bus.inbox(name):
        print(f"    from {m.sender:10s}  kind={m.kind:16s}  id={m.id}")

print("\n--- Verdict (final payload to 'user') ---")
for m in bus.inbox("user"):
    print(m.payload)

print("\n--- Isolation contract assertions ---")

# (1) Skeptic's inbox: exactly one ClaimToReview, from theorist.
skeptic_msgs = bus.inbox("skeptic")
assert len(skeptic_msgs) == 1,                    f"skeptic: {len(skeptic_msgs)} messages"
assert skeptic_msgs[0].kind == "ClaimToReview", f"skeptic kind: {skeptic_msgs[0].kind}"
assert skeptic_msgs[0].sender == "theorist",    f"skeptic sender: {skeptic_msgs[0].sender}"

# (2) Judge's inbox: exactly one ComparisonTable, from skeptic. No claim bleed-through.
judge_msgs = bus.inbox("judge")
assert len(judge_msgs) == 1,                       f"judge: {len(judge_msgs)} messages"
assert judge_msgs[0].kind == "ComparisonTable",  f"judge kind: {judge_msgs[0].kind}"
assert judge_msgs[0].sender == "skeptic",        f"judge sender: {judge_msgs[0].sender}"
assert bus.inbox("judge", kind="ClaimToReview") == [], "judge leaked claim message!"

# (3) Verdict exists and its sender is the judge.
user_msgs = bus.inbox("user")
assert len(user_msgs) == 1 and user_msgs[0].kind == "Verdict"
assert user_msgs[0].sender == "judge"

print("OK: all three isolation invariants hold.")

# Agent: Theoretical Physicist
## Task: Compute the Onsager spontaneous magnetisation |m|(T) at T = [2.0, 2.2, 2.5]. T_c = 2/ln(1+sqrt(2)) ~= 2.2692. For T < T_c, use calculator_tool to evaluate (1 - sinh(2/T)**(-4))**(1/8). For T >= T_c, the value is 0 (no tool call needed). Do NOT compute any value from memory. Procedure (follow exactly, in this order):
  step 1: calculator_tool(expression='(1 - sinh(2/2.00)**(-4))**(1/8)')  -> record as m1
  step 2: calculator_tool(expression='(1 - sinh(2/2.20)**(-4))**(1/8)')  -> record as m2
  step 3: T=2.50 is above T_c, so m3 = 0
Return a Markdown table with exactly three data rows, columns 'T' and '|m|_theorist', four decimals using m1, m2, m3.


# Agent: Theoretical Physicist
## Thought: | T    | |m|\_theorist |
|------|-------------|
| 2.00 | 0.3175      |
| 2.20 | 0.1984      |
| 2.50 | 0.0000      |
## Using tool: calculator_tool
## Tool Input: 
"{\"expression\": \"(1 - sinh(2/2.00)**(-4))**(1/8)\"}"
## Tool Output: 
0.911319377877496


# A

## 7. What we saw

**The isolation contract held.** The three `assert` lines at the end of §6 printed `OK: all three isolation invariants hold.` The bus log shows exactly three messages:

```
[52036c38]   theorist -> skeptic     kind=ClaimToReview    (116c)
[cea4b114]    skeptic -> judge       kind=ComparisonTable  (321c)
[3bd3c0cb]      judge -> user        kind=Verdict          (57c)
```

The Judge's inbox had one message, of kind `ComparisonTable`, from `skeptic`. No `ClaimToReview` leaked through. This is the first time in the course that the "what does this agent actually see?" question has a one-line answer in Python: `bus.inbox('judge')`. Compare to 2.2, where the answer was "whatever `Process.sequential` chose to pass" — a sentence, not a value.

**The end-to-end verdict matches 2.2.** Judge output:

```
VERDICT: RECONCILE. Two rows agree and one row disagrees.
```

Same `RECONCILE` label as 2.2. Same off-by-one count in the rationale (actually 1 row agreed and 2 disagreed — the Judge has now miscounted in exactly the same way on two independent runs, which is a more interesting finding than either miscount alone: it's a *reproducible* 7B-prose-counting failure, not a one-off). Label drives the downstream decision; the sentence is decorative. This is a clean A/B: different protocol, same correct adjudication.

**The Skeptic worked with a shorter prompt, but not a trivial one.** We dropped 2.2's "pretend that table is not visible to you" clause. Under A2A there is no hidden table to pretend about — the Skeptic sees a `ClaimToReview` payload wrapped in explicit markers, not a leaked Theorist context. That is the win. We *kept* the SCRATCH-lines-before-table discipline, because it guards against the 7B's tool-call-skipping failure, which is a model behaviour — not a context-bleed behaviour. **Protocol-level isolation fixes what isolation can fix; prompt discipline still earns its keep.**

**Two small discipline breaks worth naming honestly.**

First, the Skeptic called `calculator_tool` a *third* time, for T=2.50, even though the instructions said "A.3: T=2.50 is above T_c=2.2692, so mC = 0 (no tool call needed)". The tool correctly returned a complex-number error (Onsager's formula goes imaginary above T_c — the exact boundary we moved *away* from in Module 2.1 by switching to TEMPS=[2.00, 2.20, 2.50]). The Skeptic fumbled through two retries of that error, eventually wrote `mC = 0` anyway, and still produced a correct comparison table. The wasted tool calls cost ~14 s of wall time (88.7 s here vs 74.8 s in 2.2). This is prompt-adherence, not protocol.

Second, the Skeptic emitted SCRATCH lines *after* the comparison table rather than *before*, as the instructions asked. The table itself is still correct. Again: prompt adherence. A real deployment would parse the table and ignore the scratch lines anyway.

**So what did the protocol buy us?**

Three things the 2.2 version could not give:

1. **An inspectable contract.** `assert bus.inbox("judge", kind="ClaimToReview") == []` is a line of Python. If someone later wires the Theorist's claim into the Judge's inbox by accident, the notebook raises at §6. You cannot write an equivalent assertion against `Process.sequential`'s context-passing — there is no handle on it.
2. **Shorter, less defensive prompts.** The removed "pretend the table is not visible" clause was 22 words of prompt paying rent for what the protocol now enforces for free. In a larger agent network, this compounds.
3. **A substrate for cross-process / cross-agent deployment.** The shape of `a2a_run` + `MessageBus` maps one-to-one onto `python-a2a`'s `A2AClient` / `A2AServer`. When you outgrow a single laptop — when the Theorist wants GPU on one machine and the Skeptic wants a different-model runtime on another — you change `a2a_run` and *nothing else*. Exercise 2.3.3 is the migration in 40 lines.

**What the protocol did not buy us.**

- The Theorist still hallucinated. Protocol isolation is a property of *edges* in the agent graph; it does nothing about the *nodes*. Making the Theorist not-hallucinate requires what 2.2's Skeptic provided (adversarial re-derivation) or what Day-1 Module 1.4 introduced (Reflection / self-critique) or both.
- The Judge still miscounted. A 7B narrating a count of three items gets it wrong on two independent runs; the isolation layer can't fix that. If you need the rationale to be reliable, you compute it in Python from the `agree?` column before giving the model the table — the Judge becomes strictly a label-producer.
- Authentication / integrity are not implemented. Exercise 2.3.1 makes this concrete: tamper with a payload in flight, the Judge rules on the tampered thing. Typed envelopes are *not* signed envelopes; A2A proper addresses this, we did not.

## 8. Exercises and course wrap

**2.3.1 — Tamper with the payload.** Between `skeptic_turn(bus)` and `judge_turn(bus)`, mutate the Skeptic's outgoing `ComparisonTable` message by flipping every `NO` to `YES` in the payload. Re-run `judge_turn`. Does the Judge issue `ACCEPT_THEORIST` on the tampered table? The answer (yes) is the whole point of typed protocols: the bus delivers what is on it. Authentication / integrity are *additional* protocol concerns the A2A spec addresses via signed messages; we didn't implement that here, but now you know where it would go.

**2.3.2 — A fourth role: the Auditor.** Add an agent whose inbox receives every message sent on the bus (hint: extend `MessageBus.send` to also deliver to `"auditor"`). Its task: read the full bus log and produce a one-paragraph postmortem. This is the natural generalisation of `python-a2a`'s `observability` hooks.

**2.3.3 — Cross-process A2A with `python-a2a`.** Install `python-a2a`. Wrap the Skeptic in an `A2AServer` on port `8001` and the Theorist/Judge in `A2AClient`s. The turn functions' shape is identical; only `a2a_run` changes to `client.send_message(...)`. If your rewrite is longer than 40 added lines, you're over-engineering it.

---

### Course wrap

That closes **Module 2.3 — A2A** and with it the MAS Part of the course. Over two days we went:

- **SA Part.** Single agent, growing from a naked LLM through ReAct, RAG, MCP, Reflection/Reflexion, and a full Onsager use case that showed — concretely — the four failure modes a single agent cannot dig out of.
- **MAS Part.** Three specialists (2.1), then two brains contesting a claim (2.2), then the same three brains talking over a typed bus (2.3). Each module was run live against `qwen2.5:7b` on a laptop, and each module's §6/§7 names what worked and what strained.

Modules 2.4 (Planning / Coordinator patterns) and 2.5 (Evaluation harness) were scoped out on day-one planning and remain genuinely useful next reads. The course's `outline.md` will be pruned to match what we actually shipped.